In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=Warning)

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')

# Очистка пропусков для сопоставимости выборок в моделях (необходимо для инф. критериев)
vars_lfp = ['LFP', 'WA', 'FAMINC', 'WE', 'KL6', 'K618', 'CIT', 'UN']
mroz_clean = mroz_Greene.dropna(subset=vars_lfp).copy()

vars_app = ['approve', 'appinc', 'mortno', 'unem', 'dep', 'male', 'married', 'yjob', 'self']
loanapp_clean = loanapp.dropna(subset=vars_app).copy()

digits = 3

# --- Универсальная функция для вычисления всех показателей псевдо-R2 ---
def get_gof_metrics(res, model_num):
    # McFadden
    mcfadden = res.prsquared
    # Adj. McFadden
    adj_mcfadden = 1 - (res.llf - res.df_model - 1) / res.llnull
    # Cox & Snell
    cox_snell = 1 - np.exp(-res.llr / res.nobs)
    # Nagelkerke / Cragg & Uhler
    nagelkerke = cox_snell / (1 - np.exp(2 * res.llnull / res.nobs))
    # Efron
    efron = 1 - (np.sum(res.resid_response**2)) / (res.nobs * np.var(res.model.endog))
    # McKelvey & Zavoina
    var_yhat = np.var(res.fittedvalues)
    if isinstance(res.model, sm.Logit):
        mckelvey = var_yhat / (var_yhat + np.pi**2 / 3)
    else:
        mckelvey = var_yhat / (var_yhat + 1)
        
    return {
        'Model': model_num,
        'pseudo.R2': mcfadden,
        'adj.pseudo.R2': adj_mcfadden,
        'CoxSnell': cox_snell,
        'Nagelkerke': nagelkerke,
        'Efron': efron,
        'McKelveyZavoina': mckelvey
    }


# Качество подгонки

## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим несколько probit-регрессий:

* **Модель 1:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6 + \beta_5 K618 + \beta_6 CIT + \beta_7 UN + \beta_8 \log(FAMINC))$
* **Модель 2:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WE + \beta_2 KL6 + \beta_3 K618 + \beta_4 CIT + \beta_5 UN + \beta_6 \log(FAMINC))$
* **Модель 3:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6 + \beta_5 \log(FAMINC))$
* **Модель 4:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6)$

In [2]:
#| echo: false
#| warning: false
#| output: asis

f1_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
f2_lfp = 'LFP ~ WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
f3_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6 + np.log(FAMINC)'
f4_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6'

res_lfp_1 = smf.probit(f1_lfp, data=mroz_clean).fit(disp=0)
res_lfp_2 = smf.probit(f2_lfp, data=mroz_clean).fit(disp=0)
res_lfp_3 = smf.probit(f3_lfp, data=mroz_clean).fit(disp=0)
res_lfp_4 = smf.probit(f4_lfp, data=mroz_clean).fit(disp=0)

models_lfp = [res_lfp_1, res_lfp_2, res_lfp_3, res_lfp_4]
mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']
# Порядок регрессоров для таблицы LFP
reg_order_lfp = [
    'WA', 
    'I(WA ** 2)', 
    'WE', 
    'KL6', 
    'K618', 
    'CIT', 
    'UN', 
    'np.log(FAMINC)', 
    'Intercept'
]

table_lfp_base = summary_col(
    models_lfp, 
    model_names=mod_names, 
    stars=True, 
    float_format=f'%.{digits}f',
    regressor_order=reg_order_lfp
)
legend_text = "\n\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"
display(Markdown("**Результаты оценивания:**\n\n" + table_lfp_base.tables[0].to_markdown() + legend_text + "\n\n<br>\n\n**Показатели качества:**\n"))

**Результаты оценивания:**

|                | Модель 1   | Модель 2   | Модель 3   | Модель 4   |
|:---------------|:-----------|:-----------|:-----------|:-----------|
| WA             | 0.008      |            | -0.018     | -0.006     |
|                | (0.070)    |            | (0.069)    | (0.068)    |
| I(WA ** 2)     | -0.001     |            | -0.000     | -0.000     |
|                | (0.001)    |            | (0.001)    | (0.001)    |
| WE             | 0.109***   | 0.124***   | 0.109***   | 0.123***   |
|                | (0.024)    | (0.024)    | (0.024)    | (0.022)    |
| KL6            | -0.851***  | -0.621***  | -0.847***  | -0.855***  |
|                | (0.115)    | (0.098)    | (0.114)    | (0.115)    |
| K618           | -0.063     | 0.031      |            |            |
|                | (0.042)    | (0.036)    |            |            |
| CIT            | -0.128     | -0.165     |            |            |
|                | (0.107)    | (0.106)    |            |            |
| UN             | -0.011     | -0.017     |            |            |
|                | (0.016)    | (0.016)    |            |            |
| np.log(FAMINC) | 0.200*     | 0.170*     | 0.163      |            |
|                | (0.105)    | (0.103)    | (0.101)    |            |
| Intercept      | -2.005     | -2.673***  | -1.435     | -0.281     |
|                | (1.705)    | (0.957)    | (1.667)    | (1.503)    |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

<br>

**Показатели качества:**


Для каждой регрессии вычислите следующие показатели качества подгонки модели: $R^2_{pseudo}$, $adj.R^2_{pseudo}$, Cox & Snell, Nagelkerke/Cragg & Uhler, Efron, McKelvey & Zavoina. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [3]:
gof_results_lfp = [get_gof_metrics(m, i+1) for i, m in enumerate(models_lfp)]
df_gof_lfp = pd.DataFrame(gof_results_lfp).round(digits)

display(Markdown(df_gof_lfp.to_markdown(index=False)))

|   Model |   pseudo.R2 |   adj.pseudo.R2 |   CoxSnell |   Nagelkerke |   Efron |   McKelveyZavoina |
|--------:|------------:|----------------:|-----------:|-------------:|--------:|------------------:|
|       1 |       0.102 |           0.085 |      0.13  |        0.175 |   0.133 |             0.207 |
|       2 |       0.076 |           0.062 |      0.099 |        0.133 |   0.1   |             0.158 |
|       3 |       0.097 |           0.086 |      0.125 |        0.167 |   0.127 |             0.199 |
|       4 |       0.095 |           0.085 |      0.122 |        0.163 |   0.123 |             0.195 |

## approve equation #1 (logit)

Для датасета `loanapp` рассмотрим несколько logit-регрессий:

* **Модель 1:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 unem + \beta_5 dep + \beta_6 male + \beta_7 married + \beta_8 yjob + \beta_9 self)$
* **Модель 2:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 mortno + \beta_2 unem + \beta_3 dep + \beta_4 male + \beta_5 married + \beta_6 yjob + \beta_7 self)$
* **Модель 3:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 unem + \beta_5 dep + \beta_6 married)$
* **Модель 4:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 dep)$



In [4]:
#| echo: false
#| warning: false
#| output: asis

f1_app = 'approve ~ appinc + I(appinc**2) + mortno + unem + dep + male + married + yjob + self'
f2_app = 'approve ~ mortno + unem + dep + male + married + yjob + self'
f3_app = 'approve ~ appinc + I(appinc**2) + mortno + unem + dep + married'
f4_app = 'approve ~ appinc + I(appinc**2) + mortno + dep'

res_app_1 = smf.logit(f1_app, data=loanapp_clean).fit(disp=0)
res_app_2 = smf.logit(f2_app, data=loanapp_clean).fit(disp=0)
res_app_3 = smf.logit(f3_app, data=loanapp_clean).fit(disp=0)
res_app_4 = smf.logit(f4_app, data=loanapp_clean).fit(disp=0)

models_app = [res_app_1, res_app_2, res_app_3, res_app_4]
mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']
# Порядок регрессоров для таблицы approve
reg_order_app = [
    'appinc', 
    'I(appinc ** 2)', 
    'mortno', 
    'unem', 
    'dep', 
    'male', 
    'married', 
    'yjob', 
    'self', 
    'Intercept'
]

table_app_base = summary_col(
    models_app, 
    model_names=mod_names, 
    stars=True, 
    float_format=f'%.{digits}f',
    regressor_order=reg_order_app
)
display(Markdown("**Результаты оценивания:**\n\n" + table_app_base.tables[0].to_markdown() + legend_text + "\n\n<br>\n\n**Показатели качества:**\n"))

**Результаты оценивания:**

|                | Модель 1   | Модель 2   | Модель 3   | Модель 4   |
|:---------------|:-----------|:-----------|:-----------|:-----------|
| appinc         | 0.004*     |            | 0.003      | 0.004*     |
|                | (0.002)    |            | (0.002)    | (0.002)    |
| I(appinc ** 2) | -0.000**   |            | -0.000**   | -0.000**   |
|                | (0.000)    |            | (0.000)    | (0.000)    |
| mortno         | 0.707***   | 0.770***   | 0.697***   | 0.741***   |
|                | (0.175)    | (0.172)    | (0.175)    | (0.174)    |
| unem           | -0.050*    | -0.052*    | -0.061**   |            |
|                | (0.030)    | (0.029)    | (0.029)    |            |
| dep            | -0.155**   | -0.168***  | -0.157**   | -0.103*    |
|                | (0.065)    | (0.064)    | (0.065)    | (0.061)    |
| male           | -0.021     | 0.022      |            |            |
|                | (0.188)    | (0.186)    |            |            |
| married        | 0.398**    | 0.417**    | 0.398**    |            |
|                | (0.163)    | (0.162)    | (0.156)    |            |
| yjob           | -0.010     | -0.007     |            |            |
|                | (0.065)    | (0.065)    |            |            |
| self           | -0.359*    | -0.314     |            |            |
|                | (0.200)    | (0.195)    |            |            |
| Intercept      | 1.681***   | 1.860***   | 1.709***   | 1.600***   |
|                | (0.228)    | (0.193)    | (0.207)    | (0.163)    |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

<br>

**Показатели качества:**


Для каждой регрессии вычислите следующие показатели качества подгонки модели: $R^2_{pseudo}$, $adj.R^2_{pseudo}$, Cox & Snell, Nagelkerke/Cragg & Uhler, Efron, McKelvey & Zavoina. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [5]:
gof_results_app = [get_gof_metrics(m, i+1) for i, m in enumerate(models_app)]
df_gof_app = pd.DataFrame(gof_results_app).round(digits)

display(Markdown(df_gof_app.to_markdown(index=False)))

|   Model |   pseudo.R2 |   adj.pseudo.R2 |   CoxSnell |   Nagelkerke |   Efron |   McKelveyZavoina |
|--------:|------------:|----------------:|-----------:|-------------:|--------:|------------------:|
|       1 |       0.033 |           0.019 |      0.024 |        0.046 |   0.027 |             0.066 |
|       2 |       0.028 |           0.017 |      0.021 |        0.039 |   0.022 |             0.06  |
|       3 |       0.031 |           0.021 |      0.023 |        0.043 |   0.026 |             0.062 |
|       4 |       0.024 |           0.017 |      0.018 |        0.033 |   0.02  |             0.05  |

# Сравнение моделей

## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим те же спецификации probit-регрессий:

* **Модель 1:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6 + \beta_5 K618 + \beta_6 CIT + \beta_7 UN + \beta_8 \log(FAMINC))$
* **Модель 2:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WE + \beta_2 KL6 + \beta_3 K618 + \beta_4 CIT + \beta_5 UN + \beta_6 \log(FAMINC))$
* **Модель 3:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6 + \beta_5 \log(FAMINC))$
* **Модель 4:** $P(LFP=1) = \Phi(\beta_0 + \beta_1 WA + \beta_2 WA^2 + \beta_3 WE + \beta_4 KL6)$

Для каждой модели вычислите показатели информационных критериев AIC & BIC и $adj.R^2_{pseudo}$. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [6]:
#| echo: false
#| warning: false
#| output: asis

f1_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
f2_lfp = 'LFP ~ WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
f3_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6 + np.log(FAMINC)'
f4_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6'

res_lfp_1 = smf.probit(f1_lfp, data=mroz_clean).fit(disp=0)
res_lfp_2 = smf.probit(f2_lfp, data=mroz_clean).fit(disp=0)
res_lfp_3 = smf.probit(f3_lfp, data=mroz_clean).fit(disp=0)
res_lfp_4 = smf.probit(f4_lfp, data=mroz_clean).fit(disp=0)

models_lfp = [res_lfp_1, res_lfp_2, res_lfp_3, res_lfp_4]
mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']
# Порядок регрессоров для таблицы LFP
reg_order_lfp = [
    'WA', 
    'I(WA ** 2)', 
    'WE', 
    'KL6', 
    'K618', 
    'CIT', 
    'UN', 
    'np.log(FAMINC)', 
    'Intercept'
]

info_to_add = {
    'Nobs': lambda x: f"{int(x.nobs)}",
    'AIC': lambda x: f"{x.aic:.{digits}f}", 
    'BIC': lambda x: f"{x.bic:.{digits}f}", 
    'R2_adj': lambda x: f"{(1-(x.llf-x.df_model-1)/x.llnull):.{digits}f}"
}

mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']
# Порядок регрессоров для таблицы LFP
reg_order_lfp = [
    'WA', 
    'I(WA ** 2)', 
    'WE', 
    'KL6', 
    'K618', 
    'CIT', 
    'UN', 
    'np.log(FAMINC)', 
    'Intercept'
]

table_lfp = summary_col(
    models_lfp, 
    model_names=mod_names, 
    stars=True, 
    float_format=f'%.{digits}f',
    info_dict=info_to_add,
    regressor_order=reg_order_lfp
)

legend_text = "\n\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"
display(Markdown(table_lfp.tables[0].to_markdown() + legend_text))

|                | Модель 1   | Модель 2   | Модель 3   | Модель 4   |
|:---------------|:-----------|:-----------|:-----------|:-----------|
| WA             | 0.008      |            | -0.018     | -0.006     |
|                | (0.070)    |            | (0.069)    | (0.068)    |
| I(WA ** 2)     | -0.001     |            | -0.000     | -0.000     |
|                | (0.001)    |            | (0.001)    | (0.001)    |
| WE             | 0.109***   | 0.124***   | 0.109***   | 0.123***   |
|                | (0.024)    | (0.024)    | (0.024)    | (0.022)    |
| KL6            | -0.851***  | -0.621***  | -0.847***  | -0.855***  |
|                | (0.115)    | (0.098)    | (0.114)    | (0.115)    |
| K618           | -0.063     | 0.031      |            |            |
|                | (0.042)    | (0.036)    |            |            |
| CIT            | -0.128     | -0.165     |            |            |
|                | (0.107)    | (0.106)    |            |            |
| UN             | -0.011     | -0.017     |            |            |
|                | (0.016)    | (0.016)    |            |            |
| np.log(FAMINC) | 0.200*     | 0.170*     | 0.163      |            |
|                | (0.105)    | (0.103)    | (0.101)    |            |
| Intercept      | -2.005     | -2.673***  | -1.435     | -0.281     |
|                | (1.705)    | (0.957)    | (1.667)    | (1.503)    |
| AIC            | 942.680    | 965.398    | 941.392    | 941.981    |
| BIC            | 984.297    | 997.767    | 969.137    | 965.102    |
| Nobs           | 753        | 753        | 753        | 753        |
| R2_adj         | 0.085      | 0.062      | 0.086      | 0.085      |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какая модель предпочтительней по информационным критериям и $adj.R^2_{pseudo}$?

Ответ:

In [7]:
#| echo: false
#| warning: false
#| output: asis

best_lfp = pd.DataFrame({
    'Критерий': ['AIC', 'BIC', 'adj.pseudo.R2'],
    'Регрессия': [
        np.argmin([m.aic for m in models_lfp]) + 1,
        np.argmin([m.bic for m in models_lfp]) + 1,
        np.argmax([1-(m.llf-m.df_model-1)/m.llnull for m in models_lfp]) + 1
    ]
})

display(Markdown(best_lfp.to_markdown(index=False)))

| Критерий      |   Регрессия |
|:--------------|------------:|
| AIC           |           3 |
| BIC           |           4 |
| adj.pseudo.R2 |           3 |

## approve equation #1 (logit)

Для датасета `loanapp` рассмотрим несколько logit-регрессий:

* **Модель 1:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 unem + \beta_5 dep + \beta_6 male + \beta_7 married + \beta_8 yjob + \beta_9 self)$
* **Модель 2:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 mortno + \beta_2 unem + \beta_3 dep + \beta_4 male + \beta_5 married + \beta_6 yjob + \beta_7 self)$
* **Модель 3:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 unem + \beta_5 dep + \beta_6 married)$
* **Модель 4:** $P(approve=1) = \Lambda(\beta_0 + \beta_1 appinc + \beta_2 appinc^2 + \beta_3 mortno + \beta_4 dep)$

Для каждой модели вычислите показатели информационных критериев AIC & BIC и $adj.R^2_{pseudo}$. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [8]:
#| echo: false
#| warning: false
#| output: asis

f1_app = 'approve ~ appinc + I(appinc**2) + mortno + unem + dep + male + married + yjob + self'
f2_app = 'approve ~ mortno + unem + dep + male + married + yjob + self'
f3_app = 'approve ~ appinc + I(appinc**2) + mortno + unem + dep + married'
f4_app = 'approve ~ appinc + I(appinc**2) + mortno + dep'

res_app_1 = smf.logit(f1_app, data=loanapp_clean).fit(disp=0)
res_app_2 = smf.logit(f2_app, data=loanapp_clean).fit(disp=0)
res_app_3 = smf.logit(f3_app, data=loanapp_clean).fit(disp=0)
res_app_4 = smf.logit(f4_app, data=loanapp_clean).fit(disp=0)

models_app = [res_app_1, res_app_2, res_app_3, res_app_4]
mod_names = ['Модель 1', 'Модель 2', 'Модель 3', 'Модель 4']

# Порядок регрессоров для таблицы approve
reg_order_app = [
    'appinc', 
    'I(appinc ** 2)', 
    'mortno', 
    'unem', 
    'dep', 
    'male', 
    'married', 
    'yjob', 
    'self', 
    'Intercept'
]

info_to_add = {
    'Nobs': lambda x: f"{int(x.nobs)}",
    'AIC': lambda x: f"{x.aic:.{digits}f}", 
    'BIC': lambda x: f"{x.bic:.{digits}f}", 
    'R2_adj': lambda x: f"{(1-(x.llf-x.df_model-1)/x.llnull):.{digits}f}"
}

table_app = summary_col(
    models_app, 
    model_names=mod_names, 
    stars=True, 
    float_format=f'%.{digits}f',
    info_dict=info_to_add,
    regressor_order=reg_order_app
)

display(Markdown(table_app.tables[0].to_markdown() + legend_text))

|                | Модель 1   | Модель 2   | Модель 3   | Модель 4   |
|:---------------|:-----------|:-----------|:-----------|:-----------|
| appinc         | 0.004*     |            | 0.003      | 0.004*     |
|                | (0.002)    |            | (0.002)    | (0.002)    |
| I(appinc ** 2) | -0.000**   |            | -0.000**   | -0.000**   |
|                | (0.000)    |            | (0.000)    | (0.000)    |
| mortno         | 0.707***   | 0.770***   | 0.697***   | 0.741***   |
|                | (0.175)    | (0.172)    | (0.175)    | (0.174)    |
| unem           | -0.050*    | -0.052*    | -0.061**   |            |
|                | (0.030)    | (0.029)    | (0.029)    |            |
| dep            | -0.155**   | -0.168***  | -0.157**   | -0.103*    |
|                | (0.065)    | (0.064)    | (0.065)    | (0.061)    |
| male           | -0.021     | 0.022      |            |            |
|                | (0.188)    | (0.186)    |            |            |
| married        | 0.398**    | 0.417**    | 0.398**    |            |
|                | (0.163)    | (0.162)    | (0.156)    |            |
| yjob           | -0.010     | -0.007     |            |            |
|                | (0.065)    | (0.065)    |            |            |
| self           | -0.359*    | -0.314     |            |            |
|                | (0.200)    | (0.195)    |            |            |
| Intercept      | 1.681***   | 1.860***   | 1.709***   | 1.600***   |
|                | (0.228)    | (0.193)    | (0.207)    | (0.163)    |
| AIC            | 1447.463   | 1450.862   | 1444.586   | 1451.140   |
| BIC            | 1503.326   | 1495.552   | 1483.690   | 1479.072   |
| Nobs           | 1971       | 1971       | 1971       | 1971       |
| R2_adj         | 0.019      | 0.017      | 0.021      | 0.017      |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$

Какая модель предпочтительней по информационным критериям и $adj.R^2_{pseudo}$?

Ответ:

In [9]:
#| echo: false
#| warning: false
#| output: asis

best_app = pd.DataFrame({
    'Критерий': ['AIC', 'BIC', 'adj.pseudo.R2'],
    'Регрессия': [
        np.argmin([m.aic for m in models_app]) + 1,
        np.argmin([m.bic for m in models_app]) + 1,
        np.argmax([1-(m.llf-m.df_model-1)/m.llnull for m in models_app]) + 1
    ]
})

display(Markdown(best_app.to_markdown(index=False)))

| Критерий      |   Регрессия |
|:--------------|------------:|
| AIC           |           3 |
| BIC           |           4 |
| adj.pseudo.R2 |           3 |